# Step Size and Stability: Class Notebook
MATH-3515 Numerical Analysis · Module 13, Class 2

This notebook goes with the slides *Step Size and Stability*. Each part starts where a slide says **→ Notebook, Part N**.

All the code is given. Run the cells in order (Runtime → Run all works too) and read the output. Each part ends with **What to notice**.

### Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

np.set_printoptions(precision=6)
# Okabe-Ito colorblind-safe palette
OI = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]

# From Tuesday's notebook (same code)
def n_steps(T, h):
    n = round(T / h)
    if abs(n * h - T) > 1e-12:
        raise ValueError("T/h must be a whole number; choose h = 2**-k")
    return n

def euler(f, t0, x0, h, T):
    n = n_steps(T - t0, h)
    t, x = t0, x0
    for k in range(n):
        x = x + h * f(t, x)     # follow the tangent line
        t = t + h
    return x

def rk4(f, t0, x0, h, T):
    n = n_steps(T - t0, h)
    t, x = t0, x0
    for k in range(n):
        k1 = f(t, x)
        k2 = f(t + h / 2, x + h / 2 * k1)
        k3 = f(t + h / 2, x + h / 2 * k2)
        k4 = f(t + h, x + h * k3)
        x = x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        t = t + h
    return x

# The logistic model from Tuesday
r, K, x0 = 0.5, 700.0, 30.0

def f_logistic(t, x):
    return r * x * (1 - x / K)

def R_rk4(z):
    # The RK4 multiplier: one RK4 step on y' = lam y multiplies y by R_rk4(h lam)
    return 1 + z + z**2 / 2 + z**3 / 6 + z**4 / 24

print("Ready.")

## Part 1. When Euler explodes
*In class: run the next two code cells. The Try it cell is for after class.*

The model problem $y' = \lambda y$, $y(0) = 1$. One Euler step multiplies by $1 + h\lambda$: $\ y_{k+1} = (1 + h\lambda)\, y_k$. The exact value at $t_k = kh$ is $e^{\lambda t_k}$.

**Predict.** $y' = -20y$. With $h = 0.09$ and $h = 0.11$, will Euler's $y_k$ shrink or grow?

In [ ]:
lam = -20.0
part1 = {}

plt.figure(figsize=(7, 4.5))
for h, marker, style, exact_style, color in ((0.09, "o", "-", ":", OI[0]), (0.11, "s", "--", "-.", OI[1])):
    multiplier = 1 + h * lam
    y = np.zeros(11)
    y[0] = 1.0
    for k in range(10):
        y[k + 1] = multiplier * y[k]
    part1[h] = y
    print(f"h = {h}: multiplier 1 + h*lam = {multiplier:.1f}")
    print("   y_0 ... y_10:", ", ".join(f"{v:.4f}" for v in y))
    k_values = np.arange(11)
    plt.semilogy(k_values, np.abs(y), marker=marker, linestyle=style, color=color, label=f"Euler |y_k|, h = {h}")
    plt.semilogy(k_values, np.exp(lam * h * k_values), linestyle=exact_style, color=color, label=f"exact |y(t_k)|, h = {h}")
plt.xlabel("step k")
plt.ylabel("|y_k| (log scale)")
plt.title("Euler on y' = -20 y: decay or growth")
plt.grid(alpha=0.3, which="both")
plt.legend(loc="best", fontsize=8)
plt.show()

In [ ]:
part1_rk4 = {}
for h in (0.09, 0.11):
    part1_rk4[h] = abs(R_rk4(h * lam))
    print(f"h = {h}:  Euler |1 + h*lam| = {abs(1 + h * lam):.1f},   RK4 |R_rk4(h*lam)| = {part1_rk4[h]:.4f}")

**Try it: edit and run.** Change `lam` (negative) and `h`. The cell prints the multiplier, the limit $2/|\lambda|$, and whether Euler is stable.

In [ ]:
# Try it: edit and run
lam = -20
h = 0.11

multiplier = 1 + h * lam
print(f"multiplier 1 + h*lam = {multiplier:.4g}")
print(f"limit 2/|lam|        = {2 / abs(lam):.4g}")
print("stable" if abs(multiplier) < 1 else "unstable")

**What to notice.**
- The switch happens at $h = 2/|\lambda| = 0.1$: the multiplier $-0.8$ shrinks $y_k$, the multiplier $-1.2$ makes it grow.
- Both runs are inaccurate (the true $y(1) \approx$ 2e-9); only one is stable. Stability is a property of the method **and** $h$.
- RK4's limit is larger, $2.79/|\lambda| \approx 0.139$, so RK4 is stable at both step sizes ($|R|$ = 0.2854 and 0.4214).

## Part 2. Why the logistic oscillates
*In class: run the next code cell. The cobweb and the orbit diagram are* If time.

Euler on the logistic model is the fixed-point iteration $x_{k+1} = g(x_k)$ with $g(x) = x + h\,f(x)$, fixed point $K$ and $g'(K) = 1 - hr$.

**Predict.** With $h$ = 3, 4.5, 5.5 ($hr$ = 1.5, 2.25, 2.75), which runs settle at $K = 700$?

In [ ]:
def euler_iterates(h, steps=40):
    x = np.zeros(steps + 1)
    x[0] = x0
    for k in range(steps):
        x[k + 1] = x[k] + h * f_logistic(0.0, x[k])     # x_{k+1} = g(x_k)
    return x

part2 = {}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
print(f"{'h':>4s} {'hr':>5s} {'g_prime(K) = 1 - hr':>20s}   last four values")
for ax, h, marker, style, color in zip(axes, (3.0, 4.5, 5.5), ("o", "s", "^"), ("-", "-.", ":"), (OI[0], OI[1], OI[2])):
    x_k = euler_iterates(h)
    part2[h] = x_k
    print(f"{h:>4} {h * r:>5} {1 - h * r:>20}   " + ", ".join(f"{v:.1f}" for v in x_k[-4:]))
    ax.plot(np.arange(41), x_k, marker=marker, linestyle=style, color=color, markersize=4, label=f"Euler, h = {h}")
    ax.axhline(K, linestyle="--", color=OI[6], label="K = 700")
    ax.set_ylim(0, 950)
    ax.set_xlabel("step k")
    ax.set_ylabel("x_k")
    ax.set_title(f"h = {h}, hr = {h * r}")
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

### If time: cobweb diagrams (Module 3)
The graph of $g(x) = x + h\,f(x)$, the line $y = x$, and 25 iterations from $x_0 = 30$. The iteration settles where the graph crosses the line, at $K$, only when $|g'(K)| < 1$.

In [ ]:
def cobweb(ax, h, iterations=25):
    g = lambda x: x + h * f_logistic(0.0, x)
    grid = np.linspace(0, 950, 400)
    ax.plot(grid, g(grid), "-", color=OI[0], label="g(x) = x + h f(x)")
    ax.plot(grid, grid, "--", color=OI[6], label="y = x")
    xs, ys = [x0], [0.0]
    x = x0
    for _ in range(iterations):
        y = g(x)
        xs += [x, y]
        ys += [y, y]
        x = y
    ax.plot(xs, ys, ":", marker=".", color=OI[1], label="Euler iterates")
    ax.set_xlim(0, 950)
    ax.set_ylim(0, 950)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("g(x)")
    ax.set_title(f"Cobweb, h = {h}")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
cobweb(axes[0], 3.0)
cobweb(axes[1], 4.5)
plt.tight_layout()
plt.show()

### If time (after class): explore the orbit diagram
For 400 values of $hr$ between 1.5 and 3.0, take 1000 Euler steps, then plot the next 100 values as small dots. One value: Euler settles. Two values: it bounces. A smear: no pattern.

In [ ]:
def orbit_values(hr, burn_in=1000, keep=100):
    # All values of hr at once: row i holds step burn_in + 1 + i, one column per hr
    hr = np.atleast_1d(np.asarray(hr, dtype=float))
    x = np.full(hr.shape, x0)
    for _ in range(burn_in):
        x = x + hr * x * (1 - x / K)
    kept = np.zeros((keep, hr.size))
    for i in range(keep):
        x = x + hr * x * (1 - x / K)
        kept[i] = x
    return kept

hr_grid = np.linspace(1.5, 3.0, 400)
orbit = orbit_values(hr_grid)

plt.figure(figsize=(7, 4.5))
plt.plot(np.broadcast_to(hr_grid, orbit.shape).ravel(), orbit.ravel(), ".", markersize=1, color=OI[0],
         label="values after 1000 steps")
plt.axvline(2.0, linestyle="--", color=OI[1], label="h = 2/r  (hr = 2)")
plt.axvline(2.57, linestyle="--", color=OI[6], label="chaos begins  (hr = 2.57)")
plt.text(2.0, 30, " h = 2/r", color=OI[1])
plt.text(2.57, 30, " chaos begins", color=OI[6])
plt.xlabel("hr")
plt.ylabel("x_k after 1000 steps")
plt.title("Euler on the logistic model: long-run values against hr")
plt.grid(alpha=0.3)
plt.legend(loc="upper left", fontsize=8)
plt.show()

for hr_value, column in zip((2.25, 2.5), orbit_values([2.25, 2.5]).T):
    print(f"hr = {hr_value}: distinct values", np.unique(np.round(column, 1)))

**What to notice.**
- The limit is $h < 2/r = 4$, from $g'(K) = 1 - hr$ and the Module 3 test $|g'(\bar x)| < 1$.
- $h = 3$ settles at 700 ($g'(K) = -0.5$); $h = 4.5$ bounces between 500.8 and 821.5 ($g'(K) = -1.25$); $h = 5.5$ has no pattern ($g'(K) = -1.75$).
- Past the limit, the oscillation is created by the method. $K$ does not enter the limit, only $r$.

## Part 3. Aim for $e^{i\pi} = -1$
*In class: run every cell. If we are behind, skip the primer pair (it repeats the slide "Three Facts") and the `np.exp` pair (its result is on the slide "The Results").*

`1j` is Python's name for $i$. Each complex cell below is followed by its **real version**, which computes the same numbers with a 2×2 matrix. If complex numbers are new to you, read the real version of each cell: the real version is enough.

In [ ]:
points = np.array([1, 1j, -1, -1j, 1 + 0.5j])
names = ["1", "i", "-1", "-i", "1 + 0.5i"]
theta = np.linspace(0, 2 * np.pi, 200)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(np.cos(theta), np.sin(theta), "--", color=OI[0], label="unit circle")
ax.plot(points.real, points.imag, "o", color=OI[1], label="points")
for p, name in zip(points, names):
    ax.annotate(name, (p.real, p.imag), textcoords="offset points", xytext=(6, 6))
ax.set_xlim(-1.4, 1.7)
ax.set_ylim(-1.4, 1.4)
ax.set_aspect("equal")
ax.set_xlabel("real")
ax.set_ylabel("imaginary")
ax.set_title("Complex numbers are points of the plane")
ax.grid(alpha=0.3)
ax.legend(loc="lower left", fontsize=8)
plt.show()

print("abs(1 + 0.5j) =", abs(1 + 0.5j), "  (the square root of 1.25)")
print("1j * 1j =", 1j * 1j)

In [ ]:
# Real version (enough if complex numbers are new to you)
print("1 + 0.5i is the point (1, 0.5); its length is", np.hypot(1.0, 0.5))
J = np.array([[0.0, -1.0], [1.0, 0.0]])          # multiplying by i = rotating by 90 degrees
print("rotate (1, 0) by 90 degrees twice:", J @ J @ np.array([1.0, 0.0]), "  (i times i = -1)")

**Predict.** Euler with $n = 10$ steps from 1 to time $\pi$: inside or outside the unit circle?

In [ ]:
n_list = [2, 10, 100, 1000, 10000]
complex_table = {}
print(f"{'n':>6s} {'Euler y_n':>20s} {'|y_n|':>8s} {'Euler error':>12s} {'RK4 error':>10s}")
for n in n_list:
    y_euler = (1 + 1j * np.pi / n) ** n
    y_rk4 = R_rk4(1j * np.pi / n) ** n
    complex_table[n] = (y_euler, y_rk4)
    print(f"{n:>6d} {y_euler.real:>10.4f} {y_euler.imag:+.4f}i {abs(y_euler):>8.4f} "
          f"{abs(y_euler + 1):>12.3e} {abs(y_rk4 + 1):>10.3e}")

In [ ]:
# Real version (enough if complex numbers are new to you)
A_rot = np.array([[0.0, -1.0], [1.0, 0.0]])      # y' = i y  becomes  x' = A_rot x, x = (real part, imaginary part)

def f_rot(t, x):
    return A_rot @ x

start = np.array([1.0, 0.0])
real_table = {}
print(f"{'n':>6s} {'Euler x_n':>22s} {'Euler error':>12s} {'RK4 error':>10s} {'n_steps(pi, pi/n) == n':>23s}")
for n in n_list:
    h = np.pi / n
    x_euler = euler(f_rot, 0.0, start, h, np.pi)
    x_rk4 = rk4(f_rot, 0.0, start, h, np.pi)
    real_table[n] = (complex(x_euler[0], x_euler[1]), complex(x_rk4[0], x_rk4[1]))
    print(f"{n:>6d} ({x_euler[0]:>8.4f}, {x_euler[1]:>8.4f}) {np.hypot(x_euler[0] + 1, x_euler[1]):>12.3e} "
          f"{np.hypot(x_rk4[0] + 1, x_rk4[1]):>10.3e} {str(n_steps(np.pi, h) == n):>23s}")

difference = max(abs(real_table[n][i] - complex_table[n][i]) for n in n_list for i in (0, 1))
print(f"max difference from the complex table: {difference:.1e}")

In [ ]:
print("np.exp(1j*np.pi) =", np.exp(1j * np.pi))
print("np.sin(np.pi)    =", np.sin(np.pi))
print("np.pi is the double closest to pi, not pi itself; sin(np.pi) is about pi - np.pi, the gap (Module 2).")

In [ ]:
# Real version (enough if complex numbers are new to you)
c, s = np.cos(np.pi), np.sin(np.pi)
rotation = np.array([[c, -s], [s, c]])           # rotate by the angle np.pi
print("rotate (1, 0) by the angle np.pi:", rotation @ np.array([1.0, 0.0]))

In [ ]:
# The paths for n = 10, computed with the real version, one step at a time
n = 10
h = np.pi / n
euler_points, rk4_points = [start], [start]
for k in range(n):
    euler_points.append(euler(f_rot, 0.0, euler_points[-1], h, h))
    rk4_points.append(rk4(f_rot, 0.0, rk4_points[-1], h, h))
euler_points, rk4_points = np.array(euler_points), np.array(rk4_points)

theta = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(np.cos(theta), np.sin(theta), "--", color=OI[0], label="unit circle (exact path)")
ax.plot(euler_points[:, 0], euler_points[:, 1], "o-", color=OI[1], label="Euler, n = 10")
ax.plot(rk4_points[:, 0], rk4_points[:, 1], "s--", color=OI[2], label="RK4, n = 10")
ax.plot(-1, 0, "*", color=OI[6], markersize=15, label="target -1")
ax.set_aspect("equal")
ax.set_xlabel("real")
ax.set_ylabel("imaginary")
ax.set_title("Aim for -1: Euler and RK4 with n = 10")
ax.grid(alpha=0.3)
ax.legend(loc="lower left", fontsize=8)
plt.show()

**What to notice.**
- From $n = 10$ on, 10× the steps divides Euler's error by 10 (order 1) and RK4's by $10^4$ (order 4), down to rounding.
- Euler lands **outside** the circle because $|1 + i\pi/n| > 1$: $|y_{10}| = 1.6010$.
- The same numbers come from a real 2×2 matrix: the difference between the two tables is at the rounding level.

## Part 4. Oscillator, spectral radius and stability region
*In class: run the oscillator cell and the spectral radius cell. The two* If time *cells go with the backup slide "Radius Versus Norm" and the slide "The Stability Region".*

A mass on a spring, $x'' = -x$, with $v = x'$: $\ (x, v)' = A\,(x, v)$, $A = \begin{bmatrix}0&1\\-1&0\end{bmatrix}$. The exact motion is a circle, with energy $x^2 + v^2 = 1$.

In [ ]:
A = np.array([[0.0, 1.0], [-1.0, 0.0]])

def f_osc(t, y):
    return A @ y

h = 0.1
steps = 628     # ten periods. Here the number of steps is fixed, not T (T = 62.8 is not exactly 20 pi),
                # so this loop bypasses n_steps on purpose: it calls euler and rk4 one step at a time (T = h).
y_euler = np.zeros((steps + 1, 2))
y_rk4 = np.zeros((steps + 1, 2))
y_euler[0] = y_rk4[0] = [1.0, 0.0]
for k in range(steps):
    y_euler[k + 1] = euler(f_osc, 0.0, y_euler[k], h, h)
    y_rk4[k + 1] = rk4(f_osc, 0.0, y_rk4[k], h, h)
t_osc = h * np.arange(steps + 1)
energy_euler = np.sum(y_euler**2, axis=1)
energy_rk4 = np.sum(y_rk4**2, axis=1)
print(f"Euler energy after 63 steps:  {energy_euler[63]:.3f}")
print(f"Euler energy after 628 steps: {energy_euler[628]:.2f}")
print(f"RK4 energy after 628 steps:   {energy_rk4[628]:.7f}")

theta = np.linspace(0, 2 * np.pi, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].semilogy(t_osc, energy_euler, "o-", markevery=40, color=OI[1], label="Euler")
axes[0].semilogy(t_osc, energy_rk4, "s--", markevery=40, color=OI[0], label="RK4")
axes[0].set_xlabel("time t")
axes[0].set_ylabel("energy x^2 + v^2 (log scale)")
axes[0].set_title("Energy, h = 0.1, ten periods")
axes[0].grid(alpha=0.3, which="both")
axes[0].legend(loc="upper left", fontsize=8)
axes[1].plot(np.cos(theta), np.sin(theta), "--", color=OI[0], label="exact circle")
axes[1].plot(y_euler[:64, 0], y_euler[:64, 1], "o-", markersize=3, color=OI[1], label="Euler, first 63 steps")
axes[1].set_aspect("equal")
axes[1].set_xlabel("x")
axes[1].set_ylabel("v")
axes[1].set_title("Euler spirals out")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
rho = max(abs(np.linalg.eigvals(np.eye(2) + h * A)))
print(f"spectral radius rho(I + hA) = {rho:.7f}   (sqrt(1.01) = {np.sqrt(1.01):.7f})")
print(f"radius growth per period: rho^(2 pi / h) = {rho ** (2 * np.pi / h):.3f}")

### If time: radius versus norm (backup slide "Radius Versus Norm")
$G_2 = \begin{bmatrix}0.5&1\\0&0.5\end{bmatrix}$ from Module 12: its norm is above 1, its spectral radius is below 1. Which one decides what its powers do?

In [ ]:
G2 = np.array([[0.5, 1.0], [0.0, 0.5]])
rho_G2 = max(abs(np.linalg.eigvals(G2)))
print(f"||G2||_inf = {np.linalg.norm(G2, np.inf)},  rho(G2) = {rho_G2}")
g2_norms = []
print(f"{'k':>3s} {'||G2^k||_inf':>13s} {'(2k + 1)/2^k':>13s} {'rho^k':>10s}")
for k in range(1, 11):
    norm_k = np.linalg.norm(np.linalg.matrix_power(G2, k), np.inf)
    g2_norms.append(norm_k)
    print(f"{k:>3d} {norm_k:>13.6f} {(2 * k + 1) / 2**k:>13.6f} {rho_G2**k:>10.6f}")

### If time: the stability region (the slide shows it)
Put $z = h\lambda$. Euler is stable inside the disk $|1 + z| < 1$ (shaded); RK4 inside the dashed curve $|R_{\text{rk4}}(z)| = 1$.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

re_grid, im_grid = np.meshgrid(np.linspace(-3.5, 1, 451), np.linspace(-3.5, 3.5, 701))
z = re_grid + 1j * im_grid

fig, ax = plt.subplots(figsize=(6, 7))
ax.contourf(re_grid, im_grid, (abs(1 + z) < 1).astype(float), levels=[0.5, 1.5], colors=[OI[5]], alpha=0.5)
ax.contour(re_grid, im_grid, abs(1 + z), levels=[1], colors=[OI[0]], linestyles="-")
ax.contour(re_grid, im_grid, abs(R_rk4(z)), levels=[1], colors=[OI[1]], linestyles="--")
ax.plot([0.09 * -20, 0.11 * -20], [0, 0], "x", color=OI[6], markersize=10, markeredgewidth=2)
ax.annotate("h = 0.09", (-1.8, 0), textcoords="offset points", xytext=(-10, 12))
ax.annotate("h = 0.11", (-2.2, 0), textcoords="offset points", xytext=(-20, -18))
ax.plot([0, 0], [0.1, -0.1], "+", color=OI[3], markersize=12, markeredgewidth=2)
ax.annotate("z = 0.1i", (0, 0.1), textcoords="offset points", xytext=(6, 8))
ax.annotate("z = -0.1i", (0, -0.1), textcoords="offset points", xytext=(6, -16))
ax.grid(alpha=0.3)
handles = [Patch(facecolor=OI[5], alpha=0.5, edgecolor=OI[0], label="Euler stable: |1 + z| < 1"),
           Line2D([], [], color=OI[1], linestyle="--", label="RK4 boundary: |R_rk4(z)| = 1"),
           Line2D([], [], color=OI[6], marker="x", linestyle="none", markeredgewidth=2, label="z = h lam, lam = -20"),
           Line2D([], [], color=OI[3], marker="+", linestyle="none", markeredgewidth=2, label="oscillator, h = 0.1")]
ax.set_aspect("equal")
ax.set_xlabel("Re(z)")
ax.set_ylabel("Im(z)")
ax.set_title("Stability regions, z = h lam")
ax.legend(handles=handles, loc="upper left", fontsize=8)
plt.show()

region_checks = {z0: abs(R_rk4(z0)) for z0 in (-2.7, -2.9, 2.8j, 2.9j)}
for z0, value in region_checks.items():
    print(f"|R_rk4({z0})| = {value:.3f}")

**What to notice.**
- Euler's powers $(I + hA)^k$ grow like $\rho^k = 1.005^k$: the energy goes from 1 to 1.872 after 63 steps and 517.40 after 628.
- RK4 sits just inside its region on the imaginary axis, so it loses a little energy (0.99999). (*If time* cell)
- The norm of $G_2$ exceeds 1 yet its powers vanish: **the spectral radius decides**. (*If time* cell)

## Part 5 (Optional). Symplectic Euler
*Optional: goes with the backup slide "Optional: Symplectic Euler".*

Change one letter: update $x$ first, then use the **new** $x$ in the velocity update.
$$x_{k+1} = x_k + h\,v_k, \qquad v_{k+1} = v_k - h\,x_{k+1}.$$
"Symplectic" means the step keeps **area** in the $(x, v)$ plane.

In [ ]:
x_s = np.zeros(steps + 1)
v_s = np.zeros(steps + 1)
x_s[0], v_s[0] = 1.0, 0.0
for k in range(steps):
    x_s[k + 1] = x_s[k] + h * v_s[k]
    v_s[k + 1] = v_s[k] - h * x_s[k + 1]       # the new x: the one-letter change
energy_symp = x_s**2 + v_s**2

print("det of the step matrix [[1, h], [-h, 1 - h^2]] =", np.linalg.det(np.array([[1, h], [-h, 1 - h**2]])))
print(f"symplectic energy: min {energy_symp.min():.4f}, max {energy_symp.max():.4f}, final {energy_symp[-1]:.5f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].semilogy(t_osc, energy_euler, "o-", markevery=40, color=OI[1], label="Euler")
axes[0].semilogy(t_osc, energy_rk4, "s--", markevery=40, color=OI[0], label="RK4")
axes[0].semilogy(t_osc, energy_symp, "^-.", markevery=40, color=OI[2], label="symplectic Euler")
axes[0].set_xlabel("time t")
axes[0].set_ylabel("energy x^2 + v^2 (log scale)")
axes[0].set_title("Energy, h = 0.1, ten periods")
axes[0].grid(alpha=0.3, which="both")
axes[0].legend(loc="upper left", fontsize=8)
axes[1].plot(np.cos(theta), np.sin(theta), "--", color=OI[0], label="exact circle")
axes[1].plot(x_s, v_s, "-", color=OI[2], linewidth=1, label="symplectic Euler, ten periods")
axes[1].set_aspect("equal")
axes[1].set_xlabel("x")
axes[1].set_ylabel("v")
axes[1].set_title("Symplectic Euler stays near the circle")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

**What to notice.**
- The energy stays between 0.9524 and 1.0526 (final 0.99943), against 517.40 for Euler, at Euler's cost.
- It does not conserve energy exactly, but the error does not grow.

## If time: stiff problems (backup slide "Stiff Problems")
$y' = \operatorname{diag}(-1, -1000)\, y$, $y(0) = (1, 1)$. (a) Euler to $T = 1$ with $h = 1/525$ and $h = 1/475$ (both make $T/h$ a whole number). (b) `solve_ivp` to $T = 10$ with four methods.

In [ ]:
D = np.array([-1.0, -1000.0])

def f_stiff(t, y):
    return D * y

stiff_euler = {}
for n in (525, 475):
    y_end = euler(f_stiff, 0.0, np.array([1.0, 1.0]), 1 / n, 1.0)
    stiff_euler[n] = y_end
    print(f"Euler, h = 1/{n} = {1 / n:.5f}:  y(1) = ({y_end[0]:.5f}, {y_end[1]:.2e})")

stiff_ivp = {}
print(f"\n{'method':>6s} {'nfev':>6s} {'steps':>6s}")
for method in ("RK45", "Radau", "BDF", "LSODA"):
    sol = solve_ivp(f_stiff, (0, 10), [1.0, 1.0], method=method, rtol=1e-6, atol=1e-9)
    stiff_ivp[method] = (sol.nfev, len(sol.t) - 1)
    print(f"{method:>6s} {sol.nfev:>6d} {len(sol.t) - 1:>6d}")

The fast part is gone by $t = 0.01$, yet the explicit method still needs $h < 0.002$ all the way: that is stiffness.

## Summary

| Part | Slide | Takeaway |
|---|---|---|
| 1 | When Euler Explodes | Euler multiplies by $1 + h\lambda$; stable when $\|1 + h\lambda\| < 1$, that is $h < 2/\|\lambda\|$. |
| 2 | Why the Logistic Oscillates | Near $K$, $\lambda = -r$: stable for $h < 2/r$; past it, the method makes oscillations the model does not have. |
| 3 | Three Facts to The Results | Euler on $y' = iy$ spirals out because $\|1 + ih\| > 1$; the real 2×2 version gives the same numbers. |
| 4 | The Oscillator to The Stability Region | For systems, the spectral radius of $I + hA$ decides stability. |
| 5 | Optional: Symplectic Euler | A one-letter change keeps the energy bounded at Euler's cost. |

## Confidence check
Rate each statement from 1 (not yet) to 4 (confident).

- I can find the largest stable $h$ for $y' = \lambda y$ and for the logistic near $K$.
- I can explain why Euler spirals out on an oscillator.
- I can compute a spectral radius and say what it decides.
- I can tell stability from accuracy.